<a href="https://colab.research.google.com/github/mehdiabbasidev/darsman-machine-learning/blob/main/Hyperparameters_DecisionTreeClassifier2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Dataset download link:
https://drive.google.com/file/d/1hr68gp-xFtD4lnuruFO_qNDz4mwf0IEU/view?usp=sharing

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler,MinMaxScaler, OneHotEncoder

In [ ]:
file_path = '/content/drive/MyDrive/datasets/car_evaluation.csv'
df = pd.read_csv(file_path)
df.head()

,vhigh,vhigh.1,2,2.1,small,low,unacc
0,vhigh,vhigh,2,2,small,med,unacc
1,vhigh,vhigh,2,2,small,high,unacc
2,vhigh,vhigh,2,2,med,low,unacc
3,vhigh,vhigh,2,2,med,med,unacc
4,vhigh,vhigh,2,2,med,high,unacc


In [ ]:
df.columns = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class']
df.head()

,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,vhigh,2,2,small,med,unacc
1,vhigh,vhigh,2,2,small,high,unacc
2,vhigh,vhigh,2,2,med,low,unacc
3,vhigh,vhigh,2,2,med,med,unacc
4,vhigh,vhigh,2,2,med,high,unacc


In [ ]:
df.shape
df.dtypes
df.duplicated().sum()
df.nunique()

df['buying'].value_counts()
df['maint'].value_counts()
df['doors'].value_counts()
df['persons'].value_counts()
df['lug_boot'].value_counts()
df['safety'].value_counts()
df['class'].value_counts()

df.info()
df.describe()
df.isnull().sum()

class
2    1209
0     384
1      69
3      65
Name: count, dtype: int64

# LabelEncoder

In [ ]:
label_encoders = {}
for column in df.columns:
    le = LabelEncoder()
    df[column] = le.fit_transform(df[column])
    label_encoders[column] = le
df.head()

,buying,maint,doors,persons,lug_boot,safety,class
0,3,3,0,0,2,2,2
1,3,3,0,0,2,0,2
2,3,3,0,0,1,1,2
3,3,3,0,0,1,2,2
4,3,3,0,0,1,0,2


# OneHotEncoder

In [ ]:
onehot_encoder = OneHotEncoder(sparse=False)
buying_encoded = onehot_encoder.fit_transform(df[['buying']])

buying_encoded_df = pd.DataFrame(buying_encoded, columns=onehot_encoder.get_feature_names_out(['buying']))

final_df = pd.concat([df, buying_encoded_df], axis=1).drop(columns=['buying'])
df , buying_encoded_df , final_df

In [ ]:
X = df.drop(columns=['class'])
y = df['class']
X.shape , y.shape

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape , X_test.shape , y_train.shape , y_test.shape

#  fit_transform vs transform

## استانداردسازی (Standardization)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print("Mean: ", X_train.mean(axis=0))
print("Standard deviation: ", X_train.std(axis=0))

## نرمال‌سازی (Normalization)

In [ ]:
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_train , X_test

array([[0.        , 0.        , 0.66666667, 0.        , 0.        ,
        0.5       ],
       [0.66666667, 1.        , 0.66666667, 0.5       , 0.        ,
        0.5       ],
       [0.        , 0.        , 1.        , 0.        , 0.        ,
        0.        ],
       ...,
       [0.66666667, 0.33333333, 0.        , 1.        , 1.        ,
        0.5       ],
       [1.        , 1.        , 1.        , 1.        , 0.5       ,
        0.5       ],
       [0.66666667, 0.66666667, 0.66666667, 1.        , 0.        ,
        0.5       ]])

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score , recall_score , precision_score,f1_score,classification_report

dt_classifier = DecisionTreeClassifier(random_state=42)
dt_classifier.fit(X_train, y_train)
y_pred = dt_classifier.predict(X_test)

print("confusion_matrix:\n", confusion_matrix(y_test, y_pred))
print("accuracy_score:", accuracy_score(y_test, y_pred))
print("recall_score:", recall_score(y_test, y_pred,average='micro'))
print("precision_score:", precision_score(y_test, y_pred,average='micro'))
print("f1_score:", f1_score(y_test, y_pred,average='micro'))
print(f"\nClassification Report:\n{classification_report(y_test, y_pred, target_names=label_encoders['class'].classes_)}")

confusion_matrix:
 [[ 73   2   0   2]
 [  2  12   0   1]
 [  1   0 236   0]
 [  0   2   0  15]]
accuracy_score: 0.9710982658959537
recall_score: 0.9710982658959537
precision_score: 0.9710982658959537
f1_score: 0.9710982658959537

Classification Report:
              precision    recall  f1-score   support

         acc       0.96      0.95      0.95        77
        good       0.75      0.80      0.77        15
       unacc       1.00      1.00      1.00       237
       vgood       0.83      0.88      0.86        17

    accuracy                           0.97       346
   macro avg       0.89      0.91      0.90       346
weighted avg       0.97      0.97      0.97       346



# Use GridSearchCV for Best Hyperparameters

In [ ]:
dt_classifier = DecisionTreeClassifier(random_state=42)
param_grid = {
    'max_depth': [5, 10, 15, 20, 25],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': [None, 'auto', 'sqrt', 'log2']
}

In [ ]:
grid_search = GridSearchCV(estimator=dt_classifier, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [5, 10, 15, 20, 25],
                         'max_features': [None, 'auto', 'sqrt', 'log2'],
                         'min_samples_leaf': [1, 2, 4, 8],
                         'min_samples_split': [2, 5, 10, 20]},
             scoring='accuracy')

In [ ]:
best_params = grid_search.best_params_
print(f"Best Parameters : {best_params}")

Best Parameters : {'max_depth': 15, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 2}


In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
report = classification_report(y_test, y_pred, target_names=label_encoders['class'].classes_)
print(report)

              precision    recall  f1-score   support

         acc       0.96      0.95      0.95        77
        good       0.75      0.80      0.77        15
       unacc       1.00      1.00      1.00       237
       vgood       0.83      0.88      0.86        17

    accuracy                           0.97       346
   macro avg       0.89      0.91      0.90       346
weighted avg       0.97      0.97      0.97       346

